# Prediction models

Simple examples using the Exabel Python SDK: discover models, inspect configuration,
read run history, then optionally copy a model and start a run.

Follow [the setup instructions](README.md) first. Use the SDK from a checkout containing
the prediction model API additions, and launch Jupyter with EXABEL_ACCESS_TOKEN or EXABEL_API_KEY set
in its environment. The default settings only read data.

Saved outputs use illustrative data. Running the notebook shows data accessible to the configured credential.


In [1]:
import os
from copy import deepcopy

import pandas as pd
from IPython.display import display

from exabel.client.api.data_classes.prediction_model import PredictionModel
from exabel.client.api.data_classes.prediction_model_run import (
    ModelConfiguration,
    PredictionModelRun,
)
from exabel.client.api.prediction_model_api import PredictionModelApi
from exabel.client.client_config import ClientConfig

if not (os.environ.get("EXABEL_ACCESS_TOKEN") or os.environ.get("EXABEL_API_KEY")):
    raise ValueError("Set EXABEL_ACCESS_TOKEN or EXABEL_API_KEY. See README.md.")

api = PredictionModelApi(ClientConfig())

## Discover models

List one page of accessible models, most recently updated first. List responses contain
metadata; fetch a model to read its configuration. To narrow the list, set MODEL_FILTER
to an expression such as `display_name="*Revenue*"` or `folder="folders/123"`.


In [2]:
MODEL_FILTER = ""
page = api.list_models(page_size=10, order_by="update_time desc", filter=MODEL_FILTER)

models_df = pd.DataFrame(
    [
        {
            "name": model.name,
            "display_name": model.display_name,
            "folder": model.folder,
            "model_type": model.model_type,
            "update_time": model.update_time,
        }
        for model in page.results
    ]
)
display(models_df)
print(f"Total models matching the filter: {page.total_size}")

,name,display_name,folder,model_type,update_time
0,predictionModels/123,Example revenue model,folders/456,ratio_prediction,2025-01-15 12:00:00+00:00


Total models matching the filter: 1


For additional pages, pass `page.next_page_token` to `list_models` with the same
filter and ordering. `api.get_model_iterator(order_by="update_time desc", filter=MODEL_FILTER)`
follows all continuation tokens automatically.

## Inspect a model

Set MODEL_NAME to a resource name from the table, or leave it blank to inspect the
first result. Configuration keys use the API's JSON names, such as `targetSignals`.
A configuration can be readable even when it is unsupported for writes.


In [3]:
MODEL_NAME = ""
if not MODEL_NAME:
    if not page.results:
        raise ValueError("No accessible models matched. Change MODEL_FILTER or set MODEL_NAME.")
    MODEL_NAME = page.results[0].name

model = api.get_model(MODEL_NAME)
if model is None:
    raise ValueError(f"Model not found: {MODEL_NAME}")

print(model.name, model.display_name)
print("Configuration writable:", model.configuration_writable)
print("Configuration error:", model.configuration_error or "None")
display(model.configuration)

predictionModels/123 Example revenue model
Configuration writable: True
Configuration error: None


{'modelOptions': {'modelType': 'ratio_prediction',
  'parameters': {'model': 'uc_trend'}},
 'entities': [{'name': 'entityTypes/company/entities/EXAMPLE'}],
 'targetSignals': [{'signal': {'id': 1}}],
 'predictorSignals': [{'signal': {'id': 2}}],
 'trainingDuration': {'base': 'YEAR', 'multiplier': 5},
 'goal': 'PREDICT'}

## Read run history and outcomes

Runs are listed newest first. Use `api.get_run_iterator(MODEL_NAME)` to read all pages.
Fetch an exact run to inspect its saved configuration and entity outcomes.
These endpoints return configuration and execution status, not prediction time series.


In [4]:
runs_page = api.list_runs(MODEL_NAME, page_size=10)
display(
    pd.DataFrame(
        [
            {
                "name": run.name,
                "state": getattr(run.state, "name", str(run.state)),
                "active": run.active,
                "create_time": run.create_time,
                "error": run.error,
            }
            for run in runs_page.results
        ]
    )
)

,name,state,active,create_time,error
0,predictionModels/123/runs/1,SUCCEEDED,True,2025-01-15 12:00:00+00:00,


In [5]:
latest_run = None
if runs_page.results:
    latest_run = api.get_run(runs_page.results[0].name)

if latest_run is None:
    print("No run available to inspect.")
else:
    print("Run:", latest_run.name)
    print("State:", getattr(latest_run.state, "name", str(latest_run.state)))
    print("Error:", latest_run.error or "None")
    display(latest_run.model_configuration)
    display(
        pd.DataFrame(
            [
                {
                    "entity": outcome.entity,
                    "state": getattr(outcome.state, "name", str(outcome.state)),
                    "error": outcome.error,
                }
                for outcome in latest_run.entity_outcomes
            ]
        )
    )

Run: predictionModels/123/runs/1
State: SUCCEEDED
Error: None


{'modelOptions': {'modelType': 'ratio_prediction',
  'parameters': {'model': 'uc_trend'}},
 'entities': [{'name': 'entityTypes/company/entities/EXAMPLE'}],
 'targetSignals': [{'signal': {'id': 1}}],
 'predictorSignals': [{'signal': {'id': 2}}],
 'trainingDuration': {'base': 'YEAR', 'multiplier': 5},
 'goal': 'PREDICT'}

,entity,state,error
0,entityTypes/company/entities/EXAMPLE,SUCCEEDED,


## Optional: copy a model

Set RUN_WRITE_EXAMPLES to True to enable the remaining writes. This saves a new model
with a copy of the inspected configuration. It does not start a run. The source model
is unchanged. Re-running this cell creates another copy.

Choose a destination folder where the credential has write permission. Leaving
DESTINATION_FOLDER blank creates the model in the shared Analytics API folder. A writable configuration does not grant permission
to create models or use its input data.


In [6]:
RUN_WRITE_EXAMPLES = False
DESTINATION_FOLDER = ""
COPY_DISPLAY_NAME = "Notebook example"

created_model = None
if RUN_WRITE_EXAMPLES:
    if model.configuration_writable is not True or model.configuration is None:
        raise ValueError(model.configuration_error or "This configuration cannot be copied.")
    created_model = api.create_model(
        PredictionModel(
            display_name=COPY_DISPLAY_NAME,
            description="Created from the prediction model notebook",
            configuration=deepcopy(model.configuration),
        ),
        folder=DESTINATION_FOLDER or None,
    )
    print("Created:", created_model.name)
else:
    print("Write examples disabled.")

Write examples disabled.


## Optional: update the copy's description

Only fields in `update_mask` change. The description update leaves configuration untouched.


In [7]:
if RUN_WRITE_EXAMPLES and created_model is not None:
    created_model = api.update_model(
        PredictionModel(
            name=created_model.name,
            display_name=created_model.display_name,
            description="Ready for a notebook example run",
        ),
        update_mask=["description"],
    )
    print(created_model.description)

## Optional: start a run

This starts computation on the copied model with its latest configuration.
`auto_activate=False` leaves activation to the user. Re-running this cell starts another run.


In [8]:
created_run = None
if RUN_WRITE_EXAMPLES and created_model is not None:
    created_run = api.create_run(
        PredictionModelRun(
            description="Started from the prediction model notebook",
            configuration=ModelConfiguration.LATEST,
            auto_activate=False,
        ),
        model=created_model.name,
    )
    print("Created run:", created_run.name)

## Check the requested run

Re-run this cell to refresh status without starting another run. SUCCEEDED means all
recorded outcomes succeeded. MIXED indicates both successful and failed outcomes;
FAILED, CANCELLED, TIMED_OUT, and OUT_OF_MEMORY are also terminal states.


In [9]:
if created_run is not None:
    status = api.get_run(created_run.name)
    if status is None:
        print("Run not found.")
    else:
        print("Run:", status.name)
        print("State:", getattr(status.state, "name", str(status.state)))
        print("Error:", status.error or "None")
else:
    print("No run started by this notebook.")

No run started by this notebook.


## Cleanup

The public API has no endpoint to delete prediction models. Delete any example
copies in the Exabel app when finished.
